# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/13aakash/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

I rebuild the feature vector directly from the raw starter CSV (mirroring `scripts/01_prepare_features.py` line-by-line, not just calling it) so every fill and every engineered column is visible. Steps: numerics coerced with `inf`/`NaN` filled to 0, categoricals filled to `"unknown"`, rows kept only where `impressions_90d > 0` and `content_age_days >= 90`, deduplicated by `content_id`, then the label plus engineered log/flag columns are added.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, sys, subprocess
import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

raw = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(raw.shape[0], "rows,", raw.shape[1], "columns")

df = raw.copy()

numeric_fill_zero = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    "content_age_days", "age_tier_order", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct", "trend_pct",
]
for col in numeric_fill_zero:
    df[col] = pd.to_numeric(df[col], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)

categorical_cols = [
    "competition_level", "content_type", "main_intent", "provider_used", "model_used",
    "age_tier", "freshness_tier", "word_count_tier", "char_count_tier",
    "impression_tier", "position_tier", "trend_direction",
]
for col in categorical_cols:
    df[col] = df[col].fillna("unknown").astype(str).replace({"": "unknown", "nan": "unknown"})

df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
df = df.drop_duplicates(subset=["content_id"]).reset_index(drop=True)

df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])
df["has_clicks"] = (df["clicks_90d"] > 0).astype(int)
df["has_ai_sessions"] = (df["ai_sessions_90d"] > 0).astype(int)
df["measurable_opportunity"] = ((df["impressions_90d"] >= 100) & (df["sessions_90d"] > 0)).astype(int)

print(f"Prepared {len(df):,} rows (from {len(raw):,} raw rows)")
print(f"Declining rate: {df['is_declining_label'].mean():.3f}")
df.head(3)

30000 rows, 44 columns
Prepared 30,000 rows (from 30,000 raw rows)
Declining rate: 0.542


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,trend_direction,trend_pct,is_declining_label,log_impressions_90d,log_clicks_90d,log_sessions_90d,log_ai_sessions_90d,has_clicks,has_ai_sessions,measurable_opportunity
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,down,-41.4,1,8.243808,3.401197,2.890372,0.0,1,0,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,down,-57.7,1,9.636980,2.079442,2.302585,0.0,1,0,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,down,-60.9,1,9.440023,2.484907,2.484907,0.0,1,0,1


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

| Column(s) | Bucket | Available before the outcome? |
|---|---|---|
| `content_id`, `client_id` | Context (IDs) | n/a — grouping/joining/splitting only, never a feature |
| `search_volume`, `competition`, `cpc`, `content_type`, `main_intent`, `word_count`, `char_count` | Feature | Yes — content metadata, known before any outcome window |
| `provider_used`, `model_used` | Excluded | Data dictionary explicitly marks these "Not a model feature" |
| `impressions_90d`, `clicks_90d`, `sessions_90d`, etc. (90d totals) | Feature (via log-transform) | Yes — trailing measurement up to the decision point |
| `impressions_last_30d` / `impressions_prev_30d` (and clicks/sessions equivalents) | **Excluded — leakage risk** | Individually "before" the label, but together they are the exact ingredients `trend_pct` is computed from |
| `content_age_days`, `days_since_last_update`, `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct` | Feature | Yes |
| `age_tier`, `freshness_tier`, `word_count_tier`, `impression_tier`, `position_tier` | Feature | Yes — transparent buckets of the above |
| `char_count_tier` | Excluded | Redundant with `char_count`, which is already numeric |
| `trend_direction`, `trend_pct` | **Label source — never a feature** | Computed from the same 30-day windows the label uses |
| `is_declining_label` | Label | n/a — the target itself |

The code below verifies missingness patterns and cross-checks this table against the pipeline's actual `MODEL_NUMERIC_FEATURES` / `MODEL_CATEGORICAL_FEATURES`.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
missing_pct = raw.isna().mean().sort_values(ascending=False)
print("Missingness (top 15 columns):")
print((missing_pct.head(15) * 100).round(1).astype(str) + "%")

print("\nMissingness of search_volume / word_count BY content_type (checking the data-dictionary warning):")
print(raw.groupby("content_type")[["search_volume", "word_count"]].apply(lambda g: g.isna().mean().round(3)))

sys.path.insert(0, "scripts")
from ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES

print("\nOfficial numeric features the pipeline actually trains on:")
print(MODEL_NUMERIC_FEATURES)
print("\nOfficial categorical features the pipeline actually trains on:")
print(MODEL_CATEGORICAL_FEATURES)

built_numeric = set(numeric_fill_zero) | {"log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d"}
print("\nColumns I built but the pipeline deliberately excludes from the model:")
print(sorted(built_numeric - set(MODEL_NUMERIC_FEATURES)))

Missingness (top 15 columns):
provider_used        71.5%
word_count           25.7%
char_count           25.7%
word_count_tier      25.7%
char_count_tier      25.7%
model_used           19.1%
trend_pct            11.3%
competition_level     8.7%
search_volume         8.2%
cpc                   8.2%
competition           8.2%
main_intent           7.9%
scroll_rate           0.4%
content_type          0.0%
client_id             0.0%
dtype: object

Missingness of search_volume / word_count BY content_type (checking the data-dictionary warning):
                    search_volume  word_count
content_type                                 
comparison article          0.000       0.000
feedly article              1.000       0.000
keyword article             0.014       0.283

Official numeric features the pipeline actually trains on:
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d', 'days_with

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Three attacks below, each adding one more suspect column on top of the honest, pipeline-official feature set, and watching ROC AUC / Precision@50 for a suspicious jump:

1. **`trend_pct` directly** — the label's exact source.
2. **`impressions_last_30d` + `impressions_prev_30d`** — individually plausible, but together they let the model reconstruct `trend_pct` (leakage doesn't require touching the label column itself).
3. **`client_id` as a numeric code** — tests identity memorization risk, i.e. why a client-holdout split (not a random split) matters.

After running, note the actual numbers here — a big jump confirms the leak; the honest set should stay well below it.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

y = df["is_declining_label"].values

# Honest feature set — exactly what the real pipeline trains on
X_honest = df[MODEL_NUMERIC_FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0)
honest_tree = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42).fit(X_honest, y)
honest_score = honest_tree.predict_proba(X_honest)[:, 1]
print(f"HONEST features   -> ROC AUC {roc_auc_score(y, honest_score):.3f} | Precision@50 {precision_at_k(honest_score, y, 50):.3f}")

# Attack 1 — trend_pct directly
X_leak1 = X_honest.copy()
X_leak1["trend_pct"] = df["trend_pct"]
leak1_tree = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42).fit(X_leak1, y)
leak1_score = leak1_tree.predict_proba(X_leak1)[:, 1]
print(f"+ trend_pct       -> ROC AUC {roc_auc_score(y, leak1_score):.3f} | Precision@50 {precision_at_k(leak1_score, y, 50):.3f}  <- leaked")

# Attack 2 — raw 30-day windows trend_pct is derived from
X_leak2 = X_honest.copy()
X_leak2["impressions_last_30d"] = df["impressions_last_30d"]
X_leak2["impressions_prev_30d"] = df["impressions_prev_30d"]
leak2_tree = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42).fit(X_leak2, y)
leak2_score = leak2_tree.predict_proba(X_leak2)[:, 1]
print(f"+ last/prev 30d   -> ROC AUC {roc_auc_score(y, leak2_score):.3f} | Precision@50 {precision_at_k(leak2_score, y, 50):.3f}  <- leaked (reconstructs trend_pct)")

# Attack 3 — client_id as identity leakage
X_leak3 = X_honest.copy()
X_leak3["client_code"] = df["client_id"].astype("category").cat.codes
leak3_tree = DecisionTreeClassifier(max_depth=6, class_weight="balanced", random_state=42).fit(X_leak3, y)
leak3_score = leak3_tree.predict_proba(X_leak3)[:, 1]
print(f"+ client_id code  -> ROC AUC {roc_auc_score(y, leak3_score):.3f} | Precision@50 {precision_at_k(leak3_score, y, 50):.3f}  <- a client-holdout split would catch this")

HONEST features   -> ROC AUC 0.709 | Precision@50 0.940
+ trend_pct       -> ROC AUC 1.000 | Precision@50 1.000  <- leaked
+ last/prev 30d   -> ROC AUC 0.762 | Precision@50 1.000  <- leaked (reconstructs trend_pct)
+ client_id code  -> ROC AUC 0.747 | Precision@50 0.980  <- a client-holdout split would catch this


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

The code below auto-generates this list by diffing every raw column against the pipeline's official `MODEL_NUMERIC_FEATURES` / `MODEL_CATEGORICAL_FEATURES` — so it's a verified fact, not a guess.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
all_columns = set(raw.columns)
included = set(MODEL_NUMERIC_FEATURES) | set(MODEL_CATEGORICAL_FEATURES)
excluded = sorted(all_columns - included)

exclusion_reasons = {
    "content_id": "pseudonymous ID — grouping/joining only, never a feature",
    "client_id": "pseudonymous ID — used for client-holdout splits only, never a feature",
    "provider_used": "data dictionary explicitly marks this 'Not a model feature'",
    "model_used": "data dictionary explicitly marks this 'Not a model feature'",
    "trend_direction": "label source — is_declining_label is derived directly from this",
    "trend_pct": "label source — exact leakage, confirmed in §3 attack 1",
    "impressions_last_30d": "raw ingredient of trend_pct — confirmed leaky in §3 attack 2",
    "clicks_last_30d": "raw ingredient of the trend window — same risk class",
    "sessions_last_30d": "raw ingredient of the trend window — same risk class",
    "impressions_prev_30d": "raw ingredient of trend_pct — confirmed leaky in §3 attack 2",
    "clicks_prev_30d": "raw ingredient of the trend window — same risk class",
    "sessions_prev_30d": "raw ingredient of the trend window — same risk class",
    "char_count_tier": "redundant with char_count, which is already a numeric feature",
    "is_declining_label": "the target itself",
}

for col in excluded:
    reason = exclusion_reasons.get(col, "raw total already represented via its log-transformed or rate-based version")
    print(f"- {col}: {reason}")

- age_tier_order: raw total already represented via its log-transformed or rate-based version
- ai_sessions_90d: raw total already represented via its log-transformed or rate-based version
- char_count_tier: redundant with char_count, which is already a numeric feature
- clicks_90d: raw total already represented via its log-transformed or rate-based version
- clicks_last_30d: raw ingredient of the trend window — same risk class
- clicks_prev_30d: raw ingredient of the trend window — same risk class
- client_id: pseudonymous ID — used for client-holdout splits only, never a feature
- content_id: pseudonymous ID — grouping/joining only, never a feature
- engaged_sessions_90d: raw total already represented via its log-transformed or rate-based version
- impressions_90d: raw total already represented via its log-transformed or rate-based version
- impressions_last_30d: raw ingredient of trend_pct — confirmed leaky in §3 attack 2
- impressions_prev_30d: raw ingredient of trend_pct — confirm

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.